# Retrieval Augmented Generation and LangChain - Ask Eleven Madison Park Restaurant

- Build an AI Assistant that can answer questions about a specific topic (the Eleven Madison Park restaurant) based on provided documents in a database.
- This Technique is called Retrieval Augmented Generation (RAG).

## Key Objectives
- Understand the core principles of Retrieval-Augmented Generation (RAG)
- Use LangChain to streamline and orchestrate the RAG workflow.
- Load, preprocess, and split text documents for optimal performance.
- Generate high-quality embeddings using OpenAI models.
- Store and search embeddings efficiently with ChromaDB as your vector database.
- Build a powerful Retrieval-Based Question Answering (QA) Chain using LangChain.
- Design an interactive Gradio UI that displays answers alongh with source citations for transparency.

## Retrieval Augmented Generation - RAG    
- RAG is technique that combines retrieval-based systems with generative AI models.
- It's commonly used to enhance the capabilities of LLMs by intergrating external knowledge sources.  

R - Lookup the external source to retrieve the relevant information.  
A - Add the retrieved information to the user prompt as a context.  
G - Use LLM to generate a response to the user prompt with the context.

The Flow looks like this:
1. User writes a prompt or a query that is passed to an orchestrator.
2. The orchestrator sends a search query to the retriever.
3. Retriever fetches the relevant information from the knowledge sources and sends back.
4. The orchestrator augments the prompt with the context and sends it to the LLM.
5. LLM responds with the generated text which is displayed to the user via the orchestrator.

## Why Use RAG  
**Accurate Responses**: Combines retrieval with generative AI to ground outputs in factual data, minimizing errors.  
**Up-to-Date Information**: Dynamically accesses the latest content, ensuring relevance without retraining.  
**Cost-Efficient**: Reduces computational burden by relying on retrieval for context, cutting costs.  
**Customizable**: Easily tailored to specific industries and domains with custom datasets.  
**Explainable**: Increases trust by surfacing the data sources behind generated responses.

## Fine Tuning Vs. RAG  
**Fine Tuning**:
1. **What it is** - Training the base model (e.g. GPT) on a specific dataset to adapt it to a particular use case.
2. **How it Works** - Provide a labeled dataset tailored to your task. Retrain the model using this data, updating its weights.
3. **Use Case** - Useful for well-structured, high-quality datasets and niche understanding.
4. **Advantages** - Deeply integrates the knowledge from the dataset.
5. **Drawbacks** - Expensive and time-consuming. Locks the model into a specific knowledge set, limiting flexibility for dynamic or broad queries.  

**Retrieval Augmented Generation - RAG**:
1. **What it is** - Combines GPT's language generation with a retrieval mechanism that fetches external information.
2. **How it Works** - A document or database is indexed using a vector database. When queried, retrieves the most relevant chunks using similarity search. GPT generates responses by combining retrieved information with general knowledge.
3. **Use Case** - Ideal for applications requiring up-to-date or domain-specific knowledge without retraining.
4. **Advantages** - Does not require retraining the model; integrates external knowledge. Easier to maintain; data can be updated without modifying the model. Cost-effective compared to fine-tuning.
5. **Drawbacks** - Relies on quality of the retrieval system and indexed data. Retrieval may fail if documents are poorly indexed or irrelevant data is fetched. 

## LangChain 101
**What is LangChain**
- Langchain is a framwork designed to help developers build applications using large language models more efficeintly.
- Instead of just sending one prompt and getting one reply, LangChain lets you chain together different components (like prompts, memory, tools, retrieval from documents, etc.) to build more complex, smarter apps.

## LangChain Features
**Cahins**: 
You can link together multiple steps or calls to a model. Example: Ask a question -> Search Wikipedia -> Summarize -> Translate the summary.
**Agents**:
LLMs that decide which tools to use and when to use them. Think of them like smart assistants that can use calculators, databases, or serach engines as and when needed.
**Tools/Plugins**:
Connect your LLM to Google search, Python code execution, file reading, etc. Examples: "What's 347*65?" -> LLM uses a calculator tool to answer accurately.
**Memorgy**:
Keeps track of the conversation history or user preferences. Makes interactions more contextual and human-like.
**Retreival-Augmented-Generation (RAG)**:
Connect LLM to your documents (databases, websites). It retrieves relevant info from them to answer questions accurately.

## Setup, Gather RAG Tools, and Load the Data.

In [1]:
# We start by installing the libraries we need and setting up our OpenAI API key.
# This cell installs the necessary libraries. Pelase run it once.
# VERY IMPORTANT: Microsoft Visual c++ 14.0 or greater is required before running this cell.
# https://visualstudio.microsoft.com/visual-cpp-build-tools/

# the '-q' flag makes the instllation less vebose.
print("Installing necessary libraries...")
%pip install -q langchain langchain-openai openai gradio python-dotenv tiktoken langchain-community langchain-chroma chromadb
print("Libraries installed successfully!")

Installing necessary libraries...
Note: you may need to restart the kernel to use updated packages.
Libraries installed successfully!



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# Let's install and import OpenAI Packages
%pip install --upgrade openai
from openai import OpenAI

# Lets import os, which stands for "Operating System"
import os

# This will be used to load the API key from the .env file.
from dotenv import load_dotenv
load_dotenv()

# Get teh OpenAI API keys from the environment variables.
openai_api_key = os.getenv("OPENAI_API_KEY")

# Lets configure the OpenAI Client using our key
openai_client = OpenAI(api_key = openai_api_key)
print("OpenAI Cleint successfully configured!")

# LEt's view the first few characters in the key
print(openai_api_key[:10] + '...')


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


OpenAI Cleint successfully configured!
sk-proj-3Z...


In [10]:
# Lets import langchain components
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_classic.chains import RetrievalQAWithSourcesChain

In [2]:
# Define the path to your data file
# Ensure 'eleven_madison_park_data.txt' is in the same folder as this notebook 
DATA_FILE_PATH = 'eleven_madison_park_data.txt'
print(f"Data file path set to : {DATA_FILE_PATH}")

Data file path set to : eleven_madison_park_data.txt


In [3]:
# Lets load Eleven Madison Park Restaurant data, which has been scraped from the website.
# The data is saved in "eleven_madison_park_data.txt", Lanchain's TextLoader makes this easy to read.
print(f"Attempting to load data from: {DATA_FILE_PATH}")

# Initialize the TextLoader with the file path an dspecify UTF-8 encoding
# Encoding helps hanle various characters correctly.
loader = TextLoader(DATA_FILE_PATH, encoding = 'utf-8')

# Load the documents using TextLoader from LangChain, which loads the entire file as one Document object.
raw_documents = loader.load()
print(f"Successfully loaded {len(raw_documents)} document(s).")


Attempting to load data from: eleven_madison_park_data.txt
Successfully loaded 1 document(s).


In [4]:
# Lets Display a few characters of the loaded content to performa sanity check!
print(raw_documents[0].page_content[:300] + "...")

Source: https://www.elevenmadisonpark.com/
Title: Eleven Madison Park
Content:
Book on Resy
---END OF SOURCE---

Source: https://www.elevenmadisonpark.com/careers
Title: Careers — Eleven Madison Park
Content:
Join Our Team Eleven Madison Park ▾ All Businesses Eleven Madison Park Clemente Bar Daniel ...


## Splitting Documents (Chunking) With LangChain Text Splitter
Large documents are hard for AI models to process efficiently and make it difficult to find specific answers. We need to split the loaded documents into smaller, manageable "chunks". We'll use LangChain's `RecursiveCharacterTextSplitter`. 
- **Why Chunk?** : Smaller pieces are easier to embed, store, and retrieve accurately.
- `chunk_size` : Max Characters per chunk.
- `chunk_overlap` : Characters shared between consecutive chunks (helps maintain context.)

In [5]:
# Let's split the document into chunks
print("\nSplitting the loaded documents into smaller chunks...")

# Let's initialize the splitter, which tries to split the document on common separators like
# paragraphs (\n\n), sentences (.), and spaces (' ').
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,
chunk_overlap = 150,)
# split the raw document(s) into smaller document objects (chunks)
documents = text_splitter.split_documents(raw_documents)

# check if splitting produced any documents
if not documents:
    raise ValueError("Error: Splitting resutled in zero documents. Check the input file and splitter settings.")

print(f"Document split into len{documents} chunks.")


Splitting the loaded documents into smaller chunks...
Document split into len[Document(metadata={'source': 'eleven_madison_park_data.txt'}, page_content='Source: https://www.elevenmadisonpark.com/\nTitle: Eleven Madison Park\nContent:\nBook on Resy\n---END OF SOURCE---'), Document(metadata={'source': 'eleven_madison_park_data.txt'}, page_content='Source: https://www.elevenmadisonpark.com/careers\nTitle: Careers — Eleven Madison Park\nContent:'), Document(metadata={'source': 'eleven_madison_park_data.txt'}, page_content="Join Our Team Eleven Madison Park ▾ All Businesses Eleven Madison Park Clemente Bar Daniel Humm Hospitality Filter Categories Culinary Pastry Wine & Beverage Dining Room Office & Admin Other Job Types Full Time Part Time Compensation Salary Hourly Apply filters OPEN OPPORTUNITIES Staff Accountant - Part Time Eleven Madison Park Part Time • Hourly ($20 - $25) Host/Reservationist Eleven Madison Park Full Time • Hourly ($24) Sous Chef Eleven Madison Park Full Time • Salar

In [6]:
# Lets display the Pthon list contaiing the document chunks.
documents

[Document(metadata={'source': 'eleven_madison_park_data.txt'}, page_content='Source: https://www.elevenmadisonpark.com/\nTitle: Eleven Madison Park\nContent:\nBook on Resy\n---END OF SOURCE---'),
 Document(metadata={'source': 'eleven_madison_park_data.txt'}, page_content='Source: https://www.elevenmadisonpark.com/careers\nTitle: Careers — Eleven Madison Park\nContent:'),
 Document(metadata={'source': 'eleven_madison_park_data.txt'}, page_content="Join Our Team Eleven Madison Park ▾ All Businesses Eleven Madison Park Clemente Bar Daniel Humm Hospitality Filter Categories Culinary Pastry Wine & Beverage Dining Room Office & Admin Other Job Types Full Time Part Time Compensation Salary Hourly Apply filters OPEN OPPORTUNITIES Staff Accountant - Part Time Eleven Madison Park Part Time • Hourly ($20 - $25) Host/Reservationist Eleven Madison Park Full Time • Hourly ($24) Sous Chef Eleven Madison Park Full Time • Salary ($72K - $75K) Pastry Cook Eleven Madison Park Full Time • Hourly ($18 - $2

In [7]:
# Let's display an example chunk and its metadata.
print("\n---- Example chunk (Chunk 2) ----")
print(documents[2].page_content)
print("\n---- Metadata for chunk 2 ----")
print(documents[2].metadata)


---- Example chunk (Chunk 2) ----
Join Our Team Eleven Madison Park ▾ All Businesses Eleven Madison Park Clemente Bar Daniel Humm Hospitality Filter Categories Culinary Pastry Wine & Beverage Dining Room Office & Admin Other Job Types Full Time Part Time Compensation Salary Hourly Apply filters OPEN OPPORTUNITIES Staff Accountant - Part Time Eleven Madison Park Part Time • Hourly ($20 - $25) Host/Reservationist Eleven Madison Park Full Time • Hourly ($24) Sous Chef Eleven Madison Park Full Time • Salary ($72K - $75K) Pastry Cook Eleven Madison Park Full Time • Hourly ($18 - $20) Kitchen Server Eleven Madison Park Full Time • Hourly ($16) plus tips Dining Room Manager Eleven Madison Park Full Time • Salary ($72K - $75K) Porter Manager Eleven Madison Park Full Time • Salary ($70K - $75K) Senior Sous Chef Eleven Madison Park Full Time • Salary ($85K - $95K) Maitre D Eleven Madison Park Full Time • Hourly ($16) plus tips Even if you don't see the opportunity you're looking for, we would s

## Embeddings and Vector Store Creation.
**What is Embedding?**:
- In the context of Large Language Models (LLMs), embeddings refer to a way of representing words, phrases, sentences, and documents as dense vectors of numbers.
- These vectors capture the semantic meaning of the text, allowing the model to understand and work with Language more effectively.  

**Embedding Example**:
- Assume that we want to represent the words "man", "woman", "boy", and "girl" on a **semantic feature space**.
- On the X-axis, we have "gender", and on "y-axis", we have "Age". These are called **semantic features**.
- Note that two words refer to models, two to females, two to adults and two to children


![Semantic_features](Semantic_features.png)

**Lets View Embeddings in 3D Space!**
- Let's now look at the words "king", "queen", "prince", and "princes".
- While these wrods share similar gender & age attributes with "man", "woman", "boy", and "girl", they have different meanings.
- To distinguish "man" from "king", "woman" from "queen", we need to introduce a new semantic featue that sets them apart. We'll call this featue "Royalty". Let's view them in a 3D space.


![3D_space](3d_space.png)


Now, we convert our text chunks into **embeddings** (numerical vectors) using OpenAI. Similar text chunks will have similar vectors. We then store these vectors in a **vector store** (ChromaDB) for fast searching.
- **Embeddings** : Text -> Numbers (Vectors) representing meaning.
- **Vector Store** : Databse optimized for searching these vectors.

Check TensorFlow Embeddings projector (it's fun!): https://projector.tensorflow.org/ 


In [12]:
# Let's initialize our embeddings model. Note that we will use OpenAI's embedding model
print(f"Initializing OpenAI Embeddings model ...")

# Create an instance of the OpenAI Embeddings model.
# Langchain handles using the API key we loaded earlier.
embeddings = OpenAIEmbeddings(openai_api_key = openai_api_key)

print("OpenAI Embeddings model Initialized...")
# Let's create ChromaDB Vector Store.
print("\nCreating ChromaDB Vector store and embedding documents...")

# Now the chunks from 'documents' are being converted to a vector using the 'embeddings' model
# The vectors are then stored as a vector in ChromaDB.
# You could add `persist_directory = "./my_chroma_db"` to save it to disk.
# you will need to specify : (1) The list of chunked Document Objects and (2) The embedding model to use.
vector_store = Chroma.from_documents(documents = documents, embedding = embeddings)

# Verify the number of items in the store.
vector_count = vector_store._collection.count()
print(f"ChromaDB vector store created with {vector_count}")

if vector_count == 0:
    raise ValueError("Vector Store creation resulted in 0 items. Check previous steps.")

Initializing OpenAI Embeddings model ...
OpenAI Embeddings model Initialized...

Creating ChromaDB Vector store and embedding documents...
ChromaDB vector store created with 38


In [13]:
# Let's retrieve the first chunk of stored data from the vector store
stored_data = vector_store._collection.get(include=["embeddings", "documents"], limit = 1)

# Display the results 
print("First chunk text: \n", stored_data['documents'][0])
print("\nEmbeddings vector: \n", stored_data['embeddings'][0])
print(f"\nFull embedding has {len(stored_data['embeddings'][0])} dimensions.")

First chunk text: 
 Source: https://www.elevenmadisonpark.com/
Title: Eleven Madison Park
Content:
Book on Resy
---END OF SOURCE---

Embeddings vector: 
 [ 0.02330522 -0.01571015 -0.00706136 ... -0.02464633 -0.01022939
 -0.06158162]

Full embedding has 1536 dimensions.


## Testing the Retrieval
Before building full Q&A Chain, let's test if our vector store can find relevant chunks based on a smaple question. We'll use the `similarity_search` method.

In [14]:
# Let's perform a similarity search in our vector store.
print("\n--- Testing Similarity Search in Vector Stroe ---")
test_query = "What different menus are offered."
print(f"Searching for documents similar to : '{test_query}'")

# perform a similarity search. 'k=2' retrieves the top 2 most similar chunks
try:
    similar_docs = vector_store.similarity_search(test_query, k = 2)
    print(f"\nFound {len(similar_docs)} similar documents")

    for i, doc in enumerate(similar_docs):
        print(f"\n--- Document {i+10} ---")
        # Displaying the first 700 chars for brevity
        content_snippet = doc.page_content[:700].strip() + "..."
        source = doc.metadata.get("source", "Unknown Source")
        print(f'Content Snippet: {content_snippet}')
        print(f"Source : {source}")

except Exception as e:
    print(f"An Error occured during similarity search: {e}")


--- Testing Similarity Search in Vector Stroe ---
Searching for documents similar to : 'What different menus are offered.'

Found 2 similar documents

--- Document 10 ---
Content Snippet: FAQs We are located at 11 Madison Avenue, on the northeast corner of East 24th and Madison Avenue, directly across the street from Madison Square Park. We offer three menus, all 100% plant-based: Full Tasting Menu : An eight- to nine-course experience priced at $365 per guest. This menu typically lasts about two to three hours and features a mix of plated and communal dishes. 5-Course Menu : Priced at $285 per guest, this menu highlights selections from the Full Tasting Menu and lasts approximately two hours. Bar Tasting Menu : Available in our lounge for $225 per guest, this menu includes four to five courses and is designed to last around two hours. Note : These durations are estimates bas...
Source : eleven_madison_park_data.txt

--- Document 11 ---
Content Snippet: Reservations are available via 

## Building and Testing The RAG Chain Using LangChain
Now we awwemble the core RAG logic using LangChain's `RetrievalQAWithSourcesChain`. This chain combines:
1. A **Retriever**: Fetches relevant documents from our `vector_store`.
2. An **LLM** : Generates the answer based on the question and retrieved documents (we'll use OpenAI).

This specific chain type automatically handles retrieving documents, formatting them with the question for the LLM, and tracking the sources.

In [17]:
# --- 1. Define the Retriever ---
# The retriever uses the vector store to fetch documents
# We configure it to retreive the top 'k' documents.
retriever = vector_store.as_retriever(search_kwargs = {"k" : 3})
print("Retriever configured successfully from vectore store.")

# --- 2. Define the Language Model (LLM) from OpenAI ---
# Temperature cotnrols the model's creativitiy; 'temperature= 0.0' aims for more factul, less creative answer.
# You might need to specify a more powerful model, such as 'gpt-3.5-turbo-instruct'
llm = OpenAI(temperature = 0.2, openai_api_key = openai_api_key)
print("OpenAI LLM successfully configured.")

# --- 3. Create the RetreivalQAWithSourcesChain ---
# This chain type is designed specificially for Q&A with source tracking.
# chain_type = 'stuff' : Puts all retrieved text directly inot the prompt context.
# suitable if the total text withing the LLM's context limits.
# Other types like "map_reduce" hanle larger amounts of text.

qa_chain = RetrievalQAWithSourcesChain.from_chain_type(llm= llm,
chain_type = 'stuff',
retriever = retriever,

)

print("RetrievalQAWithSourcesChain created...")


Retriever configured successfully from vectore store.
OpenAI LLM successfully configured.
RetrievalQAWithSourcesChain created...


In [20]:
# --- Test the full chain ---
print("\n--- Testing The Full RAG Chain ---")
chain_test_query = 'what kind of food does Eleven Madison Park Serve?'
print(f"Query: {chain_test_query}")

# Run the query through the chain. Use invoke()for langchain >=0.1.0
# The input must be a dictionary, often with the key 'question'
try:
    result = qa_chain.invoke({"question": chain_test_query})

    # print the answer and sources from the result dictionary
    print("\n--- Answer ---")
    print(result.get("answer" , "No Answer Generated."))

    print("\n--- Sources ---")
    print(result.get("sources", "No Sources Identified."))

    # Optionally print snippets from the source documents returned.
    if "source_documents" in result:
        print("\n--- Source Document Snippets ---")
        for i, doc in enumerate(result['source_document']):
            content_snippet = doc.page_content[:250].strip()
            print(f"Doc {i+1}: {content_snippet}")
    
except Exception as e:
    print(f"\nAn error occured while runnig the chani: {e}")


--- Testing The Full RAG Chain ---
Query: what kind of food does Eleven Madison Park Serve?

--- Answer ---
 Eleven Madison Park serves a fully plant-based menu, using no animal products.


--- Sources ---
eleven_madison_park_data.txt


## Creating a Gradio Interface for RAG Chain
Let's wrap our RAG chain in a user-friendly web interface using Gradio. Users will type a question, click a button, and see the answer along with the sources the AI used.

In [22]:
# import Gradio for UI
import gradio as gr

In [ ]:
# --- Define the Function for Gradio ---
# This function takes the user's input, runs the chain and formats the output
# Ensure the 'qa_chain' variable is accessible in this scope.
def ask_elevenmadison_assistant(user_query):
    """
    Processes the user query using the RAG chain and returns formatted resutls.
    """
    print(f"Processing Gradio Query: '{user_query}'")
    if not user_query or user_query.strip() == "":
        print("---> Empty Query recieved")
        return "Please enter a question.", ""
    
    try:
        # Run the query through our RAG chain
        resutls = qa_chain.invoke({"question": user_query})

        # extract answer and sources
        answer = result.get("answer", "Sorry, I couldn't find an answer in the provided documents")
        sources = result.get("sources", "No specific sources identified.")

        # Basic formatting for sources (especially if it just returns the filename.)
        if sources == DATA_FILE_PATH:
            sources = f"Retrieved from: {DATA_FILE_PATH}"
        elif isinstance(sources, list):
            sources = ", ".join(list(set(sources)))
        
        print(f"--> Answer generated: {answer[:100].strip()}...")
        print(f"--> Sources identified: {sources}")

        return answer.strip(), sources
    
    except Exception as e:
        error_message = f"An error occured: {e}"
        print(f"--> Error during chain execution: {error_message}")
        # return error message to the user interface
        return error_message, "Error Occured"


# --- Create the Gradio Interface using Blocks API ---
print("\nSetting up Gradio interface...")

with gr.Blocks(theme = gr.themes.Soft(), title = "Eleven Madison Park Q&A Assistant") as demo:
    # Title and description for the app
    gr.Markdown(
        """
        # Eleven Madison Park - AI Q&A Assistant 💬
        Ask questions about the restaurant based on its website data.
        The AI provides answers and cites the source document.
        *(Examples: What are the menu prices? Who is the chef? Is it plant-based?)*
        """
    )

    # Input Components for the user's question
    question_input = gr.Textbox(
        label = 'Your Question:',
        placeholder = "e.g., What are the opening hours on Satudary?",
        lines = 2,
    )

    # Row Layout for the output components
    with gr.Row():
        # Output component for the generated answer (read-only)
        answer_output = gr.Textbox(label = "Answer: ",
        interactive = False, lines = 6)
        sources_output = gr.Textbox(label = "Sources: ",
        interactive = False, lines = 2)

    # Row for buttons
    with gr.Row():
        # Button to submit the question.
        submit_button = gr.Button("Ask Question", variant = "primary")
        # Clear button to reset inputs and outputs
        clear_button = gr.ClearButton(components = [question_input, answer_output, sources_output], value = 'Clear All')

    
    gr.Examples(
        examples = [
            "What are the different menu options and prices?",
            "Who is the head chef?",
            "What is Magic Farms?"
        ],
        inputs = question_input,
        cache_examples = False
    )


    # --- Connect the Submit Button to the Function ---
    # When Submit_button is clicked, call 'ask_emp_assistant'
    # pass the value from 'question_input' as input
    # put the returned values into 'answer_output' and 'sources_output' respectively
    submit_button.click(fn = ask_elevenmadison_assistant, inputs = question_input, outputs = [answer_output, sources_output])

print("\nGradion interface defined...")

# --- Launch the Gradio App ---
print("\nLaunching Gradio App.. (Stop the kernel or press Ctrl+c in terminal to quit)")
demo.launch()


Setting up Gradio interface...


C:\Users\Paco_Minha\AppData\Local\Temp\ipykernel_1436\2154719828.py:42: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme = gr.themes.Soft(), title = "Eleven Madison Park Q&A Assistant") as demo:



Gradion interface defined...

Launching Gradio App.. (Stop the kernel or press Ctrl+c in terminal to quit)
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\AI_DataScience_Learning\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Processing Gradio Query: 'What do you have in dinner'
--> Answer generated: Eleven Madison Park serves a fully plant-based menu, using no animal products....
--> Sources identified: Retrieved from: eleven_madison_park_data.txt


...

Processing Gradio Query: 'What are opening hours on sunday
'
--> Answer generated: Eleven Madison Park serves a fully plant-based menu, using no animal products....
--> Sources identified: Retrieved from: eleven_madison_park_data.txt


c:\AI_DataScience_Learning\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Processing Gradio Query: 'What are opening hours on sunday'
--> Answer generated: Eleven Madison Park serves a fully plant-based menu, using no animal products....
--> Sources identified: Retrieved from: eleven_madison_park_data.txt


c:\AI_DataScience_Learning\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Processing Gradio Query: 'Who is the head chef?'
--> Answer generated: Eleven Madison Park serves a fully plant-based menu, using no animal products....
--> Sources identified: Retrieved from: eleven_madison_park_data.txt
